<a href="https://colab.research.google.com/github/CienciaDatosUdea/005_CCA_Estudiantes/blob/main/Laboratorios/03_Lab_naive_bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Laboratorio: Naive Bayes Bernoulli para detectar spam

## Objetivo
Construir desde cero un clasificador **Naive Bayes Bernoulli** a partir de un corpus pequeño de correos.

Al terminar debes poder explicar y calcular:

$
P(Y),\qquad P(X_i\mid Y),\qquad P(X\mid Y),\qquad P(Y\mid X)
$

y comprender por qué la hipótesis

$
X_i \perp X_j\mid Y
$

reduce drásticamente la complejidad del modelo.

## Contexto

Queremos clasificar un correo como

$
Y=1:\ \text{spam}, \qquad Y=0:\ \text{normal}.
$

Usaremos cinco palabras del vocabulario:


$V=\{\text{dinero, gratis, premio, proyecto, reunion}\}.$

Cada correo se representa por

$X=(X_1,\ldots,X_5),$

donde

$
X_i=\begin{cases}
1 & \text{si aparece la palabra }i,\\
0 & \text{si no aparece.}
\end{cases}
$
Por ejemplo, "dinero gratis" se representa como $(1,1,0,0,0).$


In [5]:
corpus = [
    ("spam",   "gana dinero gratis"),
    ("spam",   "dinero gratis ahora"),
    ("spam",   "premio dinero gratis"),
    ("spam",   "gana premio ahora"),
    ("spam",   "en la reunion habra dinero gratis"),
    ("normal",  "daremos un premio despues de la reunion"),
    ("normal", "reunion de proyecto"),
    ("normal", "proyecto para mañana"),
    ("normal", "reunion mañana"),
    ("normal", "informe del proyecto"),
    ("normal", "informe del proyecto"),
]

vocabulario = ["dinero", "gratis", "premio", "proyecto", "reunion"]

## Parte 1 — Construir el vector \(X\)

1. Escribe una función `vectorizar(texto, vocabulario)` que transforme un correo en un vector binario.
2. Vectoriza todos los correos del corpus.
3. Verifica manualmente al menos dos ejemplos.

Ejemplo esperado:


$\text{"gana dinero gratis"}\longrightarrow(1,1,0,0,0)$


In [6]:
# TODO: implementa vectorizar(texto, vocabulario)
import numpy as np
x = np.array([1, 1, 1, 1, 1])

In [7]:
def vectorizar(texto, vocabulario):
    palabras_texto = texto.split()
    vector = []
    for palabra in vocabulario:
        if palabra in palabras_texto:
            vector.append(1)
        else:
            vector.append(0)
    return vector

In [15]:
def vectorizar_corpus(corpus, vocabulario):
    corpus_vectorizado = []
    for correo in corpus:
        clasificacion = 1 if correo[0] == "spam" else 0
        vector = vectorizar(correo[1], vocabulario)
        corpus_vectorizado.append((clasificacion, vector))
    return corpus_vectorizado

In [18]:
corpus_vectorizado=vectorizar_corpus(corpus, vocabulario)
for correo in corpus_vectorizado:
  print(correo)

(1, [1, 1, 0, 0, 0])
(1, [1, 1, 0, 0, 0])
(1, [1, 1, 1, 0, 0])
(1, [0, 0, 1, 0, 0])
(1, [1, 1, 0, 0, 1])
(0, [0, 0, 1, 0, 1])
(0, [0, 0, 0, 1, 1])
(0, [0, 0, 0, 1, 0])
(0, [0, 0, 0, 0, 1])
(0, [0, 0, 0, 1, 0])
(0, [0, 0, 0, 1, 0])


## Parte 2 — Calcular el prior \(P(Y)\)

Calcula:


$P(Y=\text{spam}),\qquad P(Y=\text{normal}).$


Recuerda:

$
P(Y=y)=\frac{\#\text{correos de clase }y}{\#\text{correos totales}}.
$

In [21]:
# TODO: calcula P(Y=spam) y P(Y=normal)
def p_prior(y):
  count=0
  for mensaje in corpus_vectorizado:
    if mensaje[0]==y:
      count=count+1
  return count/len(corpus)

In [30]:
print(f"P(Spam)   = {p_prior(1):.4f}")
print(f"P(Normal) = {p_prior(0):.4f}")

P(Spam)   = 0.4545
P(Normal) = 0.5455


## Parte 3 — Calcular $P(X_i=1\mid Y)$

Para cada palabra calcula su frecuencia dentro de cada clase. Por ejemplo:

$
P(X_{dinero}=1\mid Y=spam)
=\frac{\#\text{spam que contienen dinero}}{\#\text{spam}}.
$

Construye una tabla con las cinco palabras y ambas clases.

**Pregunta:** ¿qué ocurre si una palabra nunca aparece en una clase?

In [35]:
# TODO: calcula las probabilidades condicionales sin suavizado
def p_cond(indice, y):
    n_y = 0   # Correos de la clase
    n_x = 0   # Correos de la clase que tienen la palabra

    for clasificacion, vector in corpus_vectorizado:
        if clasificacion == y:
            n_y += 1
            if vector[indice] == 1:
                n_x += 1
    return n_x / n_y if n_y != 0 else 0

# Tabla
print(f"{'Palabra':<10} | {'P(Xi|Spam)':<12} | {'P(Xi|Normal)':<12}")
print("-" * 40)
for i, palabra in enumerate(vocabulario):
    p_spam = p_cond(i, 1)
    p_normal = p_cond(i, 0)
    print(f"{palabra:<10} | {p_spam:<12.4f} | {p_normal:<12.4f}")

Palabra    | P(Xi|Spam)   | P(Xi|Normal)
----------------------------------------
dinero     | 0.8000       | 0.0000      
gratis     | 0.8000       | 0.0000      
premio     | 0.4000       | 0.1667      
proyecto   | 0.0000       | 0.6667      
reunion    | 0.2000       | 0.5000      


**Respuesta** : Si una palabra nunca aparece en una clase, su probabilidad condicional $P(X_i = 1 \mid Y)$ se vuelve exactamente **0**. En la tabla anterior se puede ver que:

- $P(\text{dinero} \mid Normal) = 0$ porque ningún correo normal contiene la palabra "dinero".
- $P(\text{gratis} \mid Normal) = 0$ porque ningún correo normal contiene la palabra "gratis".
- $P(\text{proyecto} \mid Spam) = 0$ porque ningún correo spam contiene la palabra "proyecto".



**¿Por qué es un problema?**

En Naive Bayes Bernoulli, la probabilidad conjunta se calcula como un producto:

$$
P(X \mid Y) = \prod_i P(X_i = x_i \mid Y)
$$

Si una sola de esas probabilidades es 0, todo el producto se vuelve 0, sin importar cuán sospechosas sean las demás palabras del correo. Esto haría que el modelo descarte por completo una clase de forma incorrecta.

**Solución:** El suavizado de Laplace suma 1 al numerador y 2 al denominador, garantizando que ninguna probabilidad sea exactamente 0.

## Parte 4 — Suavizado de Laplace

Para evitar probabilidades exactamente iguales a cero, usa suavizado de Laplace para variables Bernoulli:


$\hat P(X_i=1\mid Y=y)=\frac{N_{iy}+1}{N_y+2},$


donde \(N_{iy}\) es el número de correos de clase \(y\) que contienen la palabra \(i\), y \(N_y\) es el número total de correos de esa clase.

Calcula de nuevo la tabla.

In [36]:
# TODO: calcula probabilidades condicionales con Laplace
def p_cond_suavizada(indice, y):
    n_y = 0   # Correos de la clase
    n_x = 0   # Correos de la clase que tienen la palabra

    for clasificacion, vector in corpus_vectorizado:
        if clasificacion == y:
            n_y += 1
            if vector[indice] == 1:
                n_x += 1
    return (n_x + 1) / (n_y + 2) if n_y != 0 else 0

# Generar la tabla
print(f"{'Palabra':<10} | {'P(Xi|Spam)':<12} | {'P(Xi|Normal)':<12}")
print("-" * 40)
for i, palabra in enumerate(vocabulario):
    p_spam = p_cond_suavizada(i, 1)
    p_normal = p_cond_suavizada(i, 0)
    print(f"{palabra:<10} | {p_spam:<12.4f} | {p_normal:<12.4f}")

Palabra    | P(Xi|Spam)   | P(Xi|Normal)
----------------------------------------
dinero     | 0.7143       | 0.1250      
gratis     | 0.7143       | 0.1250      
premio     | 0.4286       | 0.2500      
proyecto   | 0.1429       | 0.6250      
reunion    | 0.2857       | 0.5000      


## Parte 5 — Clasificar un correo nuevo

Clasifica:

> **"dinero gratis"**

Su vector es

$x=(1,1,0,0,0).$

Bajo Naive Bayes Bernoulli:

$P(x\mid Y=y)=\prod_iP(X_i=x_i\mid Y=y).$

Recuerda que para una palabra ausente:

$P(X_i=0\mid Y=y)=1-P(X_i=1\mid Y=y).$


Calcula los scores conjuntos:

$S_y=P(Y=y)P(x\mid Y=y),$

y finalmente:

$P(Y=y\mid x)=\frac{S_y}{S_{spam}+S_{normal}}.$

Decide la clase del correo.

In [55]:
# TODO: como computar likelihood, score conjunto y posterior para "dinero gratis"
def likelihood(x, y):
    prob = 1
    for i in range(len(x)):
        p_aparece = p_cond_suavizada(i, y)
        if x[i] == 1:
            prob *= p_aparece
        else:
            prob *= (1 - p_aparece)
    return prob

def score_conjunto(x):
    scores = {}
    for y in [1, 0]:
        scores[y] = p_prior(y) * likelihood(x, y)
    return scores

def p(y,x):
    scores = score_conjunto(x)
    s_y = scores[y]
    s_spam = scores[1]
    s_normal = scores[0]
    return s_y / (s_spam + s_normal)

def clasificar(mensaje):
    x = vectorizar(mensaje, vocabulario)
    p_spam   = p(1, x)
    p_normal = p(0, x)

    if p_spam > p_normal:
        return "spam", p_spam
    else:
        return "normal", p_normal

clasificar("dinero gratis")


('spam', 0.9854432517009721)

## Parte 6 — Interpretación

Responde brevemente:

**1. ¿Dónde se usa la hipótesis de independencia condicional?**

  La hipótesis de independencia condicional se usa al calcular el likelihood $P(x \mid Y)$.

  En lugar de calcular la probabilidad conjunta de todas las palabras al mismo tiempo, lo cual no es posible con pocos datos, asumimos que cada palabra es independiente de las demás dado que conocemos la clase:

  $$P(x \mid Y) = P(X_1, X_2, \ldots, X_5 \mid Y) = \prod_{i=1}^{5} P(X_i \mid Y)$$

  Es decir, en lugar de necesitar una tabla gigante con la probabilidad de cada combinación posible de palabras, solo multiplicamos las probabilidades individuales de cada palabra.

  En el código, esto se ve en la función likelihood, donde multiplicamos `prob *= p_aparece` (o `1 - p_aparece`) para cada palabra del vocabulario, sin considerar cómo interactúan entre ellas.




**2. ¿Por qué no necesitamos almacenar una probabilidad para cada uno de los \(2^5\) vectores posibles?**



Porque la hipótesis de independencia condicional reduce el número de parámetros de exponencial a lineal.

Sin la hipótesis de independencia, tendríamos que estimar la probabilidad de cada una de las $2^5 = 32$ combinaciones posibles de palabras, para cada clase. Es decir, necesitaríamos:

$$
2 \times 2^5 = 64 \text{ parámetros}
$$

Con independencia, en cambio, solo necesitamos:

- 5 probabilidades $P(X_i|Spam)$
- 5 probabilidades $P(X_i|Normal)$
- 2 priors $P(Y)$

Total: 12 parámetros en lugar de 64.

Además, cada uno de esos 12 parámetros se estima contando apariciones individuales de cada palabra, no combinaciones, por lo que necesitamos muchos menos datos para entrenar el modelo. Esto es lo que hace viable a Naive Bayes incluso con corpus pequeños como el de este laboratorio.


**3. ¿Por qué Naive Bayes se considera un modelo generativo aunque aquí lo usemos para clasificar?**

Porque modela explícitamente la distribución conjunta $P(X, Y)$ a través de dos componentes:

- El prior $P(Y)$: qué tan común es cada clase.
- El likelihood $P(X|Y)$: cómo se distribuyen las palabras dentro de cada clase.

Con esos dos componentes, el modelo puede generar datos sintéticos: bastaría con muestrear primero una clase $Y \sim P(Y)$ y luego cada palabra $X_i \sim P(X_i|Y)$ para producir un correo nuevo. Por eso se dice que aprende cómo se generan los correos, no solo cómo separarlos.

Para clasificar, aplicamos el Teorema de Bayes y convertimos ese modelo generativo en un posterior $P(Y|X)$:

$$
P(Y|X) = \frac{P(X|Y) \, P(Y)}{P(X)}
$$

Pero el modelo subyacente sigue siendo generativo, porque su objetivo original es modelar $P(X, Y)$, no directamente la frontera de decisión.
